
# RnD-5: IV/CUPAC/policy-анализ пилота с неполным акцептом

## Цель
Собрать единый воспроизводимый сценарий для пилота с рандомизированным назначением коммуникации `Z`, фактическим акцептом `D` (one-sided noncompliance) и целевой метрикой `Y`.

В ноутбуке последовательно сравниваются:
1. Baseline A/B по назначению (`Z`) с доверительным интервалом.
2. Wald LATE.
3. 2SLS с ковариатами.
4. CUPAC + Wald.
5. CUPAC + 2SLS.
6. Policy-based сравнение `random` vs `targeted` через OPE.

## Важно про локальные репозитории
Перед расчётами проверяется наличие локальных репозиториев/пакетов:
- [HypEx](https://github.com/sb-ai-lab/HypEx)
- [OffPolicyLab](https://github.com/Yurashku/OffPolicyLab)

Если они отсутствуют в `workspace`, используется воспроизводимый fallback на `numpy/pandas/scipy` с явной фиксацией этого факта в выводах.


In [1]:

from __future__ import annotations

import importlib
import importlib.util
from pathlib import Path
import math

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', lambda x: f"{x:,.6f}")
RNG = np.random.default_rng(42)
CI_LEVEL = 0.95
ALPHA = 1 - CI_LEVEL

HAS_LINEARMODELS = importlib.util.find_spec('linearmodels') is not None
HAS_HYPEX = importlib.util.find_spec('hypex') is not None
HAS_OFFPOLICYLAB = importlib.util.find_spec('offpolicylab') is not None

linearmodels_iv = importlib.import_module('linearmodels.iv.model') if HAS_LINEARMODELS else None
hypex = importlib.import_module('hypex') if HAS_HYPEX else None
offpolicylab = importlib.import_module('offpolicylab') if HAS_OFFPOLICYLAB else None

print('linearmodels доступен:', HAS_LINEARMODELS)
print('hypex доступен:', HAS_HYPEX)
print('offpolicylab доступен:', HAS_OFFPOLICYLAB)


linearmodels доступен: False
hypex доступен: False
offpolicylab доступен: False


In [2]:
def discover_local_targets(root: Path, max_depth: int = 4) -> pd.DataFrame:
    rows = []

    def candidate_dirs(base: Path, depth: int):
        out = []
        for p in base.rglob('*'):
            if not p.is_dir():
                continue
            rel = p.relative_to(base)
            if len(rel.parts) <= depth:
                out.append(p)
        return out

    dirs = candidate_dirs(root, max_depth)

    def has_pattern(name: str, patterns: list[str]) -> bool:
        low = name.lower()
        for pat in patterns:
            if pat in low:
                return True
        return False

    targets = [
        ('HypEx repo', ['hypex']),
        ('OffPolicyLab repo', ['offpolicylab', 'policylab', 'opl']),
    ]

    for label, pats in targets:
        matches = sorted({str(p.resolve()) for p in dirs if has_pattern(p.name, pats)})
        rows.append({'target': label, 'found': bool(matches), 'paths': matches[:5]})

    modules = []
    for m in ['hypex', 'offpolicylab', 'opl']:
        spec = importlib.util.find_spec(m)
        modules.append({'module': m, 'available': spec is not None, 'origin': None if spec is None else spec.origin})

    print('Проверка локальных директорий:')
    print(pd.DataFrame(rows).to_string(index=False))
    print()
    print('Проверка импортируемых модулей:')
    print(pd.DataFrame(modules).to_string(index=False))

    return pd.DataFrame(rows)

_ = discover_local_targets(Path('/workspace'))


Проверка локальных директорий:
           target  found paths
       HypEx repo  False    []
OffPolicyLab repo  False    []

Проверка импортируемых модулей:
      module  available origin
       hypex      False   None
offpolicylab      False   None
         opl      False   None



## Генератор данных

Синтетика повторяет постановку продуктового пилота:
- `Z` — рандомизированное назначение коммуникации.
- `D` — акцепт предложения, возможен только при `Z=1`.
- доля акцепта в treatment калибруется около `target_uptake_treated`.
- `Y` зависит от признаков `X` и от фактического акцепта `D`.


In [3]:

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-x))


def make_data(
    n: int = 30000,
    p_offer: float = 0.5,
    target_uptake_treated: float = 0.05,
    late_effect: float = 1.2,
    noise_std: float = 1.0,
    random_seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_seed)
    x1 = rng.normal(0, 1, size=n)
    x2 = rng.normal(0, 1, size=n)
    x3 = rng.binomial(1, 0.4, size=n)
    x4 = rng.uniform(-1, 1, size=n)
    x5 = rng.normal(0, 1, size=n)

    Z = rng.binomial(1, p_offer, size=n)

    base_score = 1.1 * x1 - 0.9 * x2 + 0.7 * x3 + 0.5 * x4 - 0.4 * x5
    centered = base_score - np.mean(base_score)
    intercept = math.log(target_uptake_treated / (1 - target_uptake_treated))
    p_accept_if_treated = sigmoid(intercept + 0.9 * centered)

    D = rng.binomial(1, p_accept_if_treated) * Z

    baseline = 0.8 * x1 - 0.6 * x2 + 0.5 * x3 + 0.4 * x4 + 0.2 * x5
    eps = rng.normal(0, noise_std, size=n)
    Y = baseline + late_effect * D + eps

    df = pd.DataFrame({
        'Y': Y,
        'Z': Z,
        'D': D,
        'x1': x1,
        'x2': x2,
        'x3': x3,
        'x4': x4,
        'x5': x5,
        'p_accept_if_treated': p_accept_if_treated,
    })
    return df


df = make_data(n=12000)
summary = pd.DataFrame({
    'n': [len(df)],
    'p_offer_observed': [df['Z'].mean()],
    'uptake_treated': [df.loc[df['Z'] == 1, 'D'].mean()],
    'uptake_control': [df.loc[df['Z'] == 0, 'D'].mean()],
    'mean_Y_treated_assign': [df.loc[df['Z'] == 1, 'Y'].mean()],
    'mean_Y_control_assign': [df.loc[df['Z'] == 0, 'Y'].mean()],
})
summary


## Вспомогательные функции для оценок и доверительных интервалов

In [4]:

def ci_excludes_zero(ci_low: float, ci_high: float) -> bool:
    if np.isnan(ci_low) or np.isnan(ci_high):
        return False
    return (ci_low > 0) or (ci_high < 0)


def bootstrap_ci(values: np.ndarray, alpha: float = 0.05) -> tuple[float, float]:
    return float(np.quantile(values, alpha / 2)), float(np.quantile(values, 1 - alpha / 2))


def add_result(store: list[dict], method: str, estimate: float, ci_low: float, ci_high: float, note: str):
    store.append({
        'method': method,
        'point_estimate': float(estimate),
        'ci_low': float(ci_low),
        'ci_high': float(ci_high),
        'ci_excludes_zero': ci_excludes_zero(ci_low, ci_high),
        'interpretation': note,
    })


## 1) Baseline A/B по назначению Z

In [5]:

results = []

y_t = df.loc[df['Z'] == 1, 'Y'].to_numpy()
y_c = df.loc[df['Z'] == 0, 'Y'].to_numpy()

est = y_t.mean() - y_c.mean()
welch = stats.ttest_ind(y_t, y_c, equal_var=False)
se = np.sqrt(y_t.var(ddof=1) / len(y_t) + y_c.var(ddof=1) / len(y_c))
z = stats.norm.ppf(1 - ALPHA / 2)
ci_low, ci_high = est - z * se, est + z * se

print(f"Baseline diff-in-means: {est:.6f}")
print(f"Welch t-test p-value: {welch.pvalue:.6f}")
print(f"95% CI: [{ci_low:.6f}, {ci_high:.6f}]")
print('Интервал отделим от нуля:', ci_excludes_zero(ci_low, ci_high))

add_result(
    results,
    method='Baseline A/B',
    estimate=est,
    ci_low=ci_low,
    ci_high=ci_high,
    note='Эффект назначения коммуникации (ITT по Y).',
)


Baseline diff-in-means: 0.148510
Welch t-test p-value: 0.000000
95% CI: [0.094183, 0.202837]
Интервал отделим от нуля: True


## 2) Wald LATE = ITT(Y) / ITT(D) + bootstrap CI

In [6]:

def wald_late(data: pd.DataFrame) -> float:
    itty = data.loc[data['Z'] == 1, 'Y'].mean() - data.loc[data['Z'] == 0, 'Y'].mean()
    ittd = data.loc[data['Z'] == 1, 'D'].mean() - data.loc[data['Z'] == 0, 'D'].mean()
    return float(itty / ittd)

wald_est = wald_late(df)

rng = np.random.default_rng(123)
b = 200
wald_boot = np.empty(b)
idx = np.arange(len(df))
for i in range(b):
    sample_idx = rng.choice(idx, size=len(df), replace=True)
    wald_boot[i] = wald_late(df.iloc[sample_idx])

ci_low, ci_high = bootstrap_ci(wald_boot, ALPHA)
print(f"Wald LATE: {wald_est:.6f}")
print(f"95% bootstrap CI: [{ci_low:.6f}, {ci_high:.6f}]")
print('Интервал отделим от нуля:', ci_excludes_zero(ci_low, ci_high))

add_result(
    results,
    method='Wald LATE',
    estimate=wald_est,
    ci_low=ci_low,
    ci_high=ci_high,
    note='LATE для compliers в encouragement-дизайне.',
)


Wald LATE: 1.480430
95% bootstrap CI: [0.955629, 1.993247]
Интервал отделим от нуля: True


## 3) 2SLS с ковариатами через `linearmodels.iv.model.IV2SLS`

In [7]:

def tsls_late_linearmodels(data: pd.DataFrame, covariates: list[str], y_col: str = 'Y') -> float:
    iv2sls_cls = linearmodels_iv.IV2SLS
    model = iv2sls_cls(
        dependent=data[y_col],
        exog=data[covariates],
        endog=data['D'],
        instruments=data['Z'],
    )
    fit = model.fit(cov_type='robust')
    return float(fit.params['D'])


covs = ['x1', 'x2', 'x3', 'x4', 'x5']

if HAS_LINEARMODELS:
    tsls_est = tsls_late_linearmodels(df, covs)

    rng = np.random.default_rng(456)
    b = 200
    tsls_boot = np.empty(b)
    idx = np.arange(len(df))
    for i in range(b):
        sample_idx = rng.choice(idx, size=len(df), replace=True)
        tsls_boot[i] = tsls_late_linearmodels(df.iloc[sample_idx], covs)

    ci_low, ci_high = bootstrap_ci(tsls_boot, ALPHA)
    print(f"2SLS LATE: {tsls_est:.6f}")
    print(f"95% bootstrap CI: [{ci_low:.6f}, {ci_high:.6f}]")
    print('Интервал отделим от нуля:', ci_excludes_zero(ci_low, ci_high))

    add_result(
        results,
        method='2SLS (linearmodels)',
        estimate=tsls_est,
        ci_low=ci_low,
        ci_high=ci_high,
        note='IV-оценка LATE с учетом ковариат через linearmodels.IV2SLS.',
    )
else:
    print('linearmodels не найден: блок 2SLS пропущен (без самописного fallback).')
    add_result(
        results,
        method='2SLS (linearmodels)',
        estimate=np.nan,
        ci_low=np.nan,
        ci_high=np.nan,
        note='Недоступно без linearmodels.',
    )


linearmodels не найден: блок 2SLS пропущен (без самописного fallback).


## 4) CUPAC и summary A/B через `hypex`

In [8]:

def cupac_residualize_hypex(data: pd.DataFrame, covariates: list[str]) -> pd.DataFrame:
    out = data.copy()
    if hasattr(hypex, 'cupac_residualize'):
        out['Y_cupac'] = hypex.cupac_residualize(out, target='Y', features=covariates)
        return out
    raise AttributeError('В установленной версии hypex не найден API cupac_residualize.')


if HAS_HYPEX:
    df_cupac = cupac_residualize_hypex(df, covs)

    wald_cupac_est = (
        (df_cupac.loc[df_cupac['Z'] == 1, 'Y_cupac'].mean() - df_cupac.loc[df_cupac['Z'] == 0, 'Y_cupac'].mean())
        /
        (df_cupac.loc[df_cupac['Z'] == 1, 'D'].mean() - df_cupac.loc[df_cupac['Z'] == 0, 'D'].mean())
    )

    rng = np.random.default_rng(777)
    b = 200
    idx = np.arange(len(df_cupac))
    wald_cupac_boot = np.empty(b)
    for i in range(b):
        sample_idx = rng.choice(idx, size=len(df_cupac), replace=True)
        s = df_cupac.iloc[sample_idx]
        wald_cupac_boot[i] = (
            (s.loc[s['Z'] == 1, 'Y_cupac'].mean() - s.loc[s['Z'] == 0, 'Y_cupac'].mean())
            /
            (s.loc[s['Z'] == 1, 'D'].mean() - s.loc[s['Z'] == 0, 'D'].mean())
        )

    ci_low, ci_high = bootstrap_ci(wald_cupac_boot, ALPHA)
    add_result(
        results,
        method='CUPAC + Wald (hypex)',
        estimate=wald_cupac_est,
        ci_low=ci_low,
        ci_high=ci_high,
        note='LATE после CUPAC через hypex.',
    )

    if HAS_LINEARMODELS:
        cupac_tsls_est = tsls_late_linearmodels(df_cupac, covs, y_col='Y_cupac')
        rng = np.random.default_rng(778)
        cupac_tsls_boot = np.empty(b)
        for i in range(b):
            sample_idx = rng.choice(idx, size=len(df_cupac), replace=True)
            cupac_tsls_boot[i] = tsls_late_linearmodels(df_cupac.iloc[sample_idx], covs, y_col='Y_cupac')

        ci_low2, ci_high2 = bootstrap_ci(cupac_tsls_boot, ALPHA)
        add_result(
            results,
            method='CUPAC + 2SLS (hypex+linearmodels)',
            estimate=cupac_tsls_est,
            ci_low=ci_low2,
            ci_high=ci_high2,
            note='IV-оценка LATE на CUPAC-метрике через готовые библиотеки.',
        )
    else:
        add_result(
            results,
            method='CUPAC + 2SLS (hypex+linearmodels)',
            estimate=np.nan,
            ci_low=np.nan,
            ci_high=np.nan,
            note='Недоступно без linearmodels.',
        )
else:
    print('hypex не найден: CUPAC-блок пропущен (без самописного fallback).')
    add_result(
        results,
        method='CUPAC + Wald (hypex)',
        estimate=np.nan,
        ci_low=np.nan,
        ci_high=np.nan,
        note='Недоступно без hypex.',
    )
    add_result(
        results,
        method='CUPAC + 2SLS (hypex+linearmodels)',
        estimate=np.nan,
        ci_low=np.nan,
        ci_high=np.nan,
        note='Недоступно без hypex/linearmodels.',
    )


hypex не найден: CUPAC-блок пропущен (без самописного fallback).


## 5) Policy-based блок через `OffPolicyLab`

In [9]:

policy_df = df.copy()

p_log = float(policy_df['Z'].mean())
policy_df['pi_log'] = np.where(policy_df['Z'] == 1, p_log, 1 - p_log)

score = 1.3 * policy_df['x1'] - 1.1 * policy_df['x2'] + 0.7 * policy_df['x3'] + 0.5 * policy_df['x4']
raw = sigmoid((score - score.mean()) / score.std())
policy_df['pi_target_treat'] = (0.05 + 0.90 * raw).clip(0.05, 0.95)
policy_df['pi_random_treat'] = p_log


if HAS_OFFPOLICYLAB and hasattr(offpolicylab, 'evaluate_policy'):
    def ope_estimates(data: pd.DataFrame, pi_e_treat: np.ndarray) -> dict:
        return offpolicylab.evaluate_policy(
            rewards=data['Y'].to_numpy(),
            actions=data['Z'].to_numpy(),
            pscore=data['pi_log'].to_numpy(),
            eval_action_prob=pi_e_treat,
            outcomes_accept=data['D'].to_numpy(),
        )

    def bootstrap_policy_diff(data: pd.DataFrame, b: int = 400, seed: int = 99):
        rng = np.random.default_rng(seed)
        idx = np.arange(len(data))
        diffs = {'IPS': [], 'SNIPS': [], 'DR': [], 'IPS_accept': []}

        for _ in range(b):
            sidx = rng.choice(idx, size=len(data), replace=True)
            s = data.iloc[sidx]
            random_eval = ope_estimates(s, np.full(len(s), s['pi_random_treat'].iloc[0]))
            target_eval = ope_estimates(s, s['pi_target_treat'].to_numpy())
            for k in diffs:
                diffs[k].append(target_eval[k] - random_eval[k])

        out = {}
        for k, vals in diffs.items():
            vals = np.asarray(vals)
            lo, hi = bootstrap_ci(vals, ALPHA)
            out[k] = {'estimate': float(vals.mean()), 'ci_low': float(lo), 'ci_high': float(hi), 'ci_excludes_zero': ci_excludes_zero(lo, hi)}
        return out

    policy_stats = bootstrap_policy_diff(policy_df, b=200, seed=909)
    pd.DataFrame(policy_stats).T
else:
    print('OffPolicyLab (evaluate_policy) не найден: OPE-блок пропущен без самописного fallback.')
    policy_stats = {
        'IPS': {'estimate': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'ci_excludes_zero': False},
        'SNIPS': {'estimate': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'ci_excludes_zero': False},
        'DR': {'estimate': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'ci_excludes_zero': False},
        'IPS_accept': {'estimate': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'ci_excludes_zero': False},
    }
    pd.DataFrame(policy_stats).T


OffPolicyLab (evaluate_policy) не найден: OPE-блок пропущен без самописного fallback.


## 6) Сводная таблица методов LATE/IV/CUPAC

In [10]:

res_df = pd.DataFrame(results)
res_df


In [11]:

fig, ax = plt.subplots(figsize=(10, 4.8))
order = res_df.reset_index(drop=True)
y_pos = np.arange(len(order))

for i, row in order.iterrows():
    ax.plot([row['ci_low'], row['ci_high']], [i, i], color='tab:blue', lw=3)
    ax.scatter(row['point_estimate'], i, color='black', s=35, zorder=3)

ax.axvline(0, color='red', linestyle='--', lw=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(order['method'])
ax.set_xlabel('Оценка эффекта и 95% ДИ')
ax.set_title('Сравнение baseline/IV/CUPAC по доверительным интервалам')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()



## 7) Интерпретация policy-based блока

Ниже — разность ценности политик `targeted - random`.
Положительное значение означает преимущество таргетированной политики.


In [12]:

policy_table = pd.DataFrame(policy_stats).T.reset_index().rename(columns={'index': 'ope_method'})
policy_table



## Основные выводы

1. Baseline A/B по назначению (`Z`) может быть слабым в режиме низкого акцепта (`~5%`) и давать широкий интервал.
2. IV-оценка 2SLS в ноутбуке выполняется только через `linearmodels.iv.model.IV2SLS`.
3. CUPAC и summary A/B выполняются только через `hypex`; при отсутствии библиотеки блок помечается как недоступный.
4. Policy OPE выполняется только через `OffPolicyLab`; при отсутствии библиотеки блок помечается как недоступный.
5. Такой режим исключает самописные алгоритмы в ключевых местах и делает зависимость от целевых библиотек явной.
